# Lesson 4.1 — Defining Evaluation Metrics

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Define metrics across accuracy, latency, cost, safety, and security
- Choose metrics that reflect the business success criterion
- Encode metrics as functions you can run on system outputs

**You will produce:** A `metrics` module computing accuracy, latency, cost, safety, and security scores.

## Phase 4 Overview

Domain 4 (16%) is how you *prove* a Claude system works and keep it working: metrics, eval datasets, mixed-method tests, A/B testing, diagnosis of failures, and optimization of token/latency/cost. This phase builds a reusable evaluation harness — the single most valuable artifact for both the exam and real delivery — and connects back to the Phase 3 observability data.

By the end of this phase, you will have built:

- A multi-dimensional metrics module (accuracy/latency/cost/safety/security)
- A labeled eval dataset + scorer with mixed-method checks
- An A/B testing harness with a simple significance check
- A failure-diagnosis decision tree
- A token/latency/cost optimizer

### Prerequisites

- Completed Phases 1-3
- Python 3; `pip install anthropic` optional
- Comfort with basic statistics (rates, percentiles)

## Concepts

You cannot optimize what you do not measure. A production Claude eval spans five dimensions:

- **Accuracy/quality** — does the output match the gold answer or satisfy a rubric?
- **Latency** — p50/p95 response time against the SLA.
- **Cost** — tokens x price per request and per outcome.
- **Safety** — rate of harmful, biased, or policy-violating outputs.
- **Security** — resistance to prompt injection / data exfiltration.

Metrics must trace back to the **business success metric** from Phase 1 (e.g. '>=95% agreement with nurses'). A system can be fast and cheap yet unsafe — report all five so trade-offs are explicit. We implement each as a function so they can run automatically in CI and in the A/B harness later this phase.

> **Reference:** [Anthropic — Evaluating model performance](https://docs.anthropic.com/en/docs/test-and-evaluate/develop-tests)

In [ ]:
# !pip install anthropic
import os, json, time, random

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        # Deterministic mock: echo a label when the prompt asks for classification
        return f"[MOCK {model}] {prompt[:60]}"
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Implement accuracy, latency, and cost metrics

We start with the three quantitative metrics. Accuracy compares predictions to gold labels; latency summarizes timings; cost derives from tokens and placeholder prices.

In [ ]:
def accuracy(preds, gold):
    assert len(preds)==len(gold)
    return sum(p==g for p,g in zip(preds,gold))/len(gold)

def latency_summary(times_ms):
    s = sorted(times_ms)
    return {"p50": s[len(s)//2], "p95": s[int(0.95*(len(s)-1))]}

PRICE_PER_1K = {"in":0.003, "out":0.015}  # placeholders; verify at https://docs.anthropic.com
def cost(in_tok, out_tok):
    return in_tok/1000*PRICE_PER_1K['in'] + out_tok/1000*PRICE_PER_1K['out']

print('accuracy', accuracy(['approve','deny','needs_info'], ['approve','deny','deny']))
print('latency', latency_summary([400,900,1200,2600,1500]))
print('cost/req $', round(cost(1200,180),5))

## Code-Along: Step 2 — Add safety and security metrics

Safety and security are rates over labeled probes. We compute the fraction of outputs that violate a safety rule and the fraction of injection probes that succeeded (lower is better).

In [ ]:
def safety_violation_rate(flags):
    # flags: list of bools, True = output violated a safety/policy rule
    return sum(flags)/len(flags) if flags else 0.0

def injection_success_rate(probe_results):
    # probe_results: True = injection succeeded (bad)
    return sum(probe_results)/len(probe_results) if probe_results else 0.0

print('safety violation', safety_violation_rate([False,False,True,False]))
print('injection success', injection_success_rate([False,False,False,True,False]))

## Exercise

Build a single `evaluate_system()` that returns a dict with all five metrics given predictions, gold, timings, token counts, safety flags, and injection results. Then write a one-line `gate()` that returns pass/fail against thresholds tied to the Phase 1 success metric (e.g. accuracy>=0.95, p95<=20000ms, safety_violation==0).

In [ ]:
# TODO: def evaluate_system(...) -> {accuracy, latency, cost, safety, security}
# TODO: def gate(metrics) -> (bool, reasons) against explicit thresholds

## Key Takeaways

- Evaluate five dimensions: accuracy, latency, cost, safety, security.
- Tie every metric back to the Phase 1 business success criterion.
- Encode metrics as functions so they run in CI and the A/B harness.
- Fast and cheap is worthless if unsafe — always report safety and security.

---

**Next:** [Lesson 4.2 — Eval Datasets and Mixed-Method Test Frameworks](./lesson-4.2-eval-datasets-frameworks.ipynb)